In [6]:
import struct
import numpy as np

def load_idx_images(path):
    with open(path, 'rb') as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8)
        images = images.reshape(num, rows, cols)
    return images

def load_idx_labels(path):
    with open(path, 'rb') as f:
        magic, num = struct.unpack(">II", f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

def rotate(images):
    images = np.transpose(images, (0, 2, 1))
    images = np.flip(images, axis=2)
    return images

train_images = load_idx_images('../data/original/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('../data/original/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('../data/original/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('../data/original/emnist-byclass-test-labels-idx1-ubyte')

train_images = rotate(train_images)
test_images = rotate(test_images)

mapping = {}
# Digits
for i in range(10):
    mapping[i] = str(i)
# Lowercase letters
for i in range(26):
    mapping[10 + i] = chr(65 + i)
# Uppercase letters
for i in range(26):
    mapping[36 + i] = chr(97 + i)

In [8]:
import numpy as np
from scipy.ndimage import rotate
import matplotlib.pyplot as plt
import os

def balance_dataset_augmentation(train_images, train_labels, mapping):
    num_classes = len(mapping)
    
    class_counts = []
    for c in range(num_classes):
        count = np.sum(train_labels == c)
        class_counts.append(count)
    
    target_count = int(np.mean(class_counts))
    print(f"Target count per class: {target_count}")
    
    balanced_images = []
    balanced_labels = []
    
    for c in range(num_classes):
        idx = np.where(train_labels == c)[0]
        class_images = train_images[idx]
        current_count = len(class_images)
        
        print(f"Class {c} ({list(mapping.values())[c]}): {current_count} -> {target_count}")
        
        if current_count >= target_count:
            selected_idx = np.random.choice(current_count, target_count, replace=False)
            balanced_images.append(class_images[selected_idx])
            balanced_labels.extend([c] * target_count)
        else:
            balanced_images.append(class_images)
            balanced_labels.extend([c] * current_count)
            
            need_more = target_count - current_count

            augmented = []
            angles = np.linspace(-12, 12, 25) 
            
            while len(augmented) < need_more:
                img_idx = np.random.randint(0, current_count)
                img = class_images[img_idx]
                
                angle = np.random.choice(angles)
                
                rotated = rotate(img, angle, reshape=False, order=1, mode='constant', cval=0)
                rotated = np.clip(rotated, 0, 255).astype(np.uint8)
                
                augmented.append(rotated)
            
            augmented = np.array(augmented[:need_more])
            balanced_images.append(augmented)
            balanced_labels.extend([c] * need_more)
    
    balanced_images = np.concatenate(balanced_images, axis=0)
    balanced_labels = np.array(balanced_labels)
    
    shuffle_idx = np.random.permutation(len(balanced_labels))
    balanced_images = balanced_images[shuffle_idx]
    balanced_labels = balanced_labels[shuffle_idx]
    
    print(f"\nOriginal dataset size: {len(train_images)}")
    print(f"Balanced dataset size: {len(balanced_images)}")
    
    return balanced_images, balanced_labels

def add_realistic_noise(train_images, train_labels, noise_level='medium'):
    noisy_images = train_images.copy().astype(np.float32)
    
    if noise_level == 'light':
        gaussian_std = 5
        salt_pepper_prob = 0.005
        texture_intensity = 0.1
    elif noise_level == 'medium':
        gaussian_std = 10
        salt_pepper_prob = 0.01
        texture_intensity = 0.2
    else: 
        gaussian_std = 15
        salt_pepper_prob = 0.02
        texture_intensity = 0.3
    
    for i in range(len(noisy_images)):
        img = noisy_images[i]
        
        gaussian = np.random.normal(0, gaussian_std, img.shape)
        img = img + gaussian
        
        salt_pepper = np.random.random(img.shape)
        img[salt_pepper < salt_pepper_prob] = 255  
        img[salt_pepper > (1 - salt_pepper_prob)] = 0  
        
        texture = np.random.uniform(-texture_intensity * 255, 
                                   texture_intensity * 255, 
                                   img.shape)
        background_mask = img < 50
        img[background_mask] += texture[background_mask]
        
        if np.random.random() < 0.3: 
            from scipy.ndimage import gaussian_filter
            img = gaussian_filter(img, sigma=0.5)
        
        brightness_factor = np.random.uniform(0.8, 1.2)
        img = img * brightness_factor
        
        noisy_images[i] = np.clip(img, 0, 255)
    
    noisy_images = noisy_images.astype(np.uint8)
    
    return noisy_images, train_labels

def visualize_results(original_images, original_labels, 
                     balanced_images, balanced_labels,
                     noisy_images, noisy_labels,
                     mapping, sample_class=0):
    fig, axes = plt.subplots(3, 5, figsize=(15, 9))
    
    orig_idx = np.where(original_labels == sample_class)[0][:5]
    bal_idx = np.where(balanced_labels == sample_class)[0][:5]
    noisy_idx = np.where(noisy_labels == sample_class)[0][:5]
    
    for i in range(5):
        axes[0, i].imshow(original_images[orig_idx[i]], cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title(f'Original\nClass: {list(mapping.values())[sample_class]}', 
                               fontsize=10, fontweight='bold')
    
    for i in range(5):
        axes[1, i].imshow(balanced_images[bal_idx[i]], cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Balanced\n(with augmentation)', 
                               fontsize=10, fontweight='bold')

    for i in range(5):
        axes[2, i].imshow(noisy_images[noisy_idx[i]], cmap='gray')
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_title('With Realistic Noise', 
                               fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('comparison_methods.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    num_classes = len(mapping)
    
    orig_counts = [np.sum(original_labels == c) for c in range(num_classes)]
    axes[0].bar(range(num_classes), orig_counts, color='steelblue')
    axes[0].set_title('Original Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Count')
    axes[0].axhline(np.mean(orig_counts), color='red', linestyle='--', 
                    label=f'Mean: {int(np.mean(orig_counts))}')
    axes[0].legend()
    
    bal_counts = [np.sum(balanced_labels == c) for c in range(num_classes)]
    axes[1].bar(range(num_classes), bal_counts, color='green')
    axes[1].set_title('Balanced Distribution', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Count')
    axes[1].axhline(np.mean(bal_counts), color='red', linestyle='--',
                    label=f'Mean: {int(np.mean(bal_counts))}')
    axes[1].legend()
    
    noisy_counts = [np.sum(noisy_labels == c) for c in range(num_classes)]
    axes[2].bar(range(num_classes), noisy_counts, color='orange')
    axes[2].set_title('Noisy Distribution (unchanged)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Class')
    axes[2].set_ylabel('Count')
    axes[2].axhline(np.mean(noisy_counts), color='red', linestyle='--',
                    label=f'Mean: {int(np.mean(noisy_counts))}')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig('distribution_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

def write_idx_images(filename, images):
    num_images = images.shape[0]
    num_rows = images.shape[1]
    num_cols = images.shape[2]
    
    with open(filename, 'wb') as f:
        f.write(struct.pack('>I', 2051))
        f.write(struct.pack('>I', num_images))
        f.write(struct.pack('>I', num_rows))
        f.write(struct.pack('>I', num_cols))
        f.write(images.tobytes())
    
    print(f"Written {num_images} images to {filename}")


def write_idx_labels(filename, labels):    
    num_labels = len(labels)
    
    with open(filename, 'wb') as f:
        f.write(struct.pack('>I', 2049))
        f.write(struct.pack('>I', num_labels))
        f.write(labels.astype(np.uint8).tobytes())
    
    print(f"Written {num_labels} labels to {filename}")


def export_balanced_dataset(train_images, train_labels, test_images, test_labels, 
                           output_dir='../data/balanced'):
    os.makedirs(output_dir, exist_ok=True)
    
    balanced_train_images, balanced_train_labels = balance_dataset_augmentation(
        train_images, train_labels, mapping
    )
    
    write_idx_images(
        f'{output_dir}/emnist-byclass-train-images-idx3-ubyte',
        balanced_train_images
    )
    write_idx_labels(
        f'{output_dir}/emnist-byclass-train-labels-idx1-ubyte',
        balanced_train_labels
    )
    write_idx_images(
        f'{output_dir}/emnist-byclass-test-images-idx3-ubyte',
        test_images
    )
    write_idx_labels(
        f'{output_dir}/emnist-byclass-test-labels-idx1-ubyte',
        test_labels
    )
    
    print("BALANCED DATASET")
    print(f"Train: {len(balanced_train_images)} images")
    print(f"Test: {len(test_images)} images")


def export_noisy_dataset(train_images, train_labels, test_images, test_labels,
                        output_dir='../data/noisy', noise_level='medium'):
    os.makedirs(output_dir, exist_ok=True)

    noisy_train_images, noisy_train_labels = add_realistic_noise(
        train_images, train_labels, noise_level=noise_level
    )
    
    noisy_test_images, noisy_test_labels = add_realistic_noise(
        test_images, test_labels, noise_level='light'
    )
    
    write_idx_images(
        f'{output_dir}/emnist-byclass-train-images-idx3-ubyte',
        noisy_train_images
    )
    write_idx_labels(
        f'{output_dir}/emnist-byclass-train-labels-idx1-ubyte',
        noisy_train_labels
    )
    write_idx_images(
        f'{output_dir}/emnist-byclass-test-images-idx3-ubyte',
        noisy_test_images
    )
    write_idx_labels(
        f'{output_dir}/emnist-byclass-test-labels-idx1-ubyte',
        noisy_test_labels
    )
    
    print("NOISY DATASET")
    print(f"Train: {len(noisy_train_images)} images")
    print(f"Test: {len(noisy_test_images)} images")


def export_combined_dataset(train_images, train_labels, test_images, test_labels,
                           output_dir='../data/combined', noise_level='medium'):
    os.makedirs(output_dir, exist_ok=True)
    
    balanced_train_images, balanced_train_labels = balance_dataset_augmentation(
        train_images, train_labels, mapping
    )
    
    noisy_balanced_train_images, noisy_balanced_train_labels = add_realistic_noise(
        balanced_train_images, balanced_train_labels, noise_level=noise_level
    )
    
    noisy_test_images, noisy_test_labels = add_realistic_noise(
        test_images, test_labels, noise_level='light'
    )
    
    write_idx_images(
        f'{output_dir}/emnist-byclass-train-images-idx3-ubyte',
        noisy_balanced_train_images
    )
    write_idx_labels(
        f'{output_dir}/emnist-byclass-train-labels-idx1-ubyte',
        noisy_balanced_train_labels
    )
    write_idx_images(
        f'{output_dir}/emnist-byclass-test-images-idx3-ubyte',
        noisy_test_images
    )
    write_idx_labels(
        f'{output_dir}/emnist-byclass-test-labels-idx1-ubyte',
        noisy_test_labels
    )
    
    print("COMBINED DATASET")
    print(f"Train: {len(noisy_balanced_train_images)} images")
    print(f"Test: {len(noisy_test_images)} images")

export_balanced_dataset(train_images, train_labels, test_images, test_labels,
                        output_dir='../data/balanced')

export_noisy_dataset(train_images, train_labels, test_images, test_labels,
                    output_dir='../data/noisy', noise_level='medium')

export_combined_dataset(train_images, train_labels, test_images, test_labels,
                       output_dir='../data/combined', noise_level='medium')

Target count per class: 11256
Class 0 (0): 34585 -> 11256
Class 1 (1): 38374 -> 11256
Class 2 (2): 34203 -> 11256
Class 3 (3): 35143 -> 11256
Class 4 (4): 33535 -> 11256
Class 5 (5): 31416 -> 11256
Class 6 (6): 34232 -> 11256
Class 7 (7): 35754 -> 11256
Class 8 (8): 33946 -> 11256
Class 9 (9): 33847 -> 11256
Class 10 (A): 6407 -> 11256
Class 11 (B): 3878 -> 11256
Class 12 (C): 10094 -> 11256
Class 13 (D): 4562 -> 11256
Class 14 (E): 4934 -> 11256
Class 15 (F): 9182 -> 11256
Class 16 (G): 2517 -> 11256
Class 17 (H): 3152 -> 11256
Class 18 (I): 11946 -> 11256
Class 19 (J): 3762 -> 11256
Class 20 (K): 2468 -> 11256
Class 21 (L): 5076 -> 11256
Class 22 (M): 9002 -> 11256
Class 23 (N): 8237 -> 11256
Class 24 (O): 24983 -> 11256
Class 25 (P): 8347 -> 11256
Class 26 (Q): 2605 -> 11256
Class 27 (R): 5073 -> 11256
Class 28 (S): 20764 -> 11256
Class 29 (T): 9820 -> 11256
Class 30 (U): 12602 -> 11256
Class 31 (V): 4637 -> 11256
Class 32 (W): 4695 -> 11256
Class 33 (X): 2771 -> 11256
Class 34 (Y):